In [ ]:
import pandas as pd
import os 
import numpy as np
import geopandas as gpd

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

import seaborn as sns

from shapely import wkt

from PIL import Image

pd.set_option('display.max_columns', None)


# Read in dataset

In [ ]:
d = gpd.read_file("/home/csutter/DRIVE-clean/weather_events/data/nws_warnings/nws_all_warnings_cleaned.gpkg")
# ("/home/csutter/DRIVE-clean/NWS_warnings/data/nws_warnings/nws_all_warnings_cleaned.gpkg")

d.head(4)

# add timedelta duration col back (using duration_sec col)
# note that you have to do this w/ every gdf
d["duration"] = pd.to_timedelta(d["duration_sec"], unit="s")

# may take ~40 seconds

# Explore what warnings there are (no need to run)
# print(np.unique(d["name"]))
# Yes: 'Blizzard Warning', 'Dense Fog Advisory', 'Lake Effect Snow Warning', 'Snow Squall Warning', 'Winter Storm Warning', 'Winter Storm Watch', 'Winter Weather Advisory'
# Maybe: 'Cold Weather Advisory', 'Ice Storm Warning'

eventsofint = ['Blizzard Warning', 'Dense Fog Advisory', 'Lake Effect Snow Warning', 'Snow Squall Warning', 'Winter Storm Warning', 'Winter Storm Watch', 'Winter Weather Advisory']

devents = d[d["name"].isin(eventsofint)]

# Grab some basic counts for reference

In [ ]:
print(len(devents)) # note that this isn't the number of instances to run for inference. Because 1) events elements are a range of time, so need to run multiple 5-min instances within one 'event', so this makes *more* instances, 2) If there an event that spans multiple regions, there is likely overlapping times of the warning. These will end up being duplicate because when we prepare the inference instances to run (below) we run the entire state for every time where there was an active snow squall warning, regardless of where the warning was (cleaner to maintain consistency in inference runs by running every instance statewide)

display(devents.groupby(["name"]).count())

In [ ]:
# some basic info, no need to run

squall = d[d["name"]=="Snow Squall Warning"]
print("number of snow squall entries (note tha these aren't unique events, b/c one event may have multiple locations/datetimes associated with it. E.g. if squall Saratoga warning was issue at 10am, and Albany warning was at 1015am. Or even if Albany was also at 10am, same time as the Saratoga issued time, it would still be two entries.")
print(len(squall))
print("unique locations since 2022 that have had snow squalls")
print(len(np.unique(squall["loc_desc"])))
print("unique datetimes since 2022 that have had snow squalls")
print(len(np.unique(squall["issued"])))

blizzard = d[d["name"]=="Blizzard Warning"]
print(len(blizzard))

ws_warn = d[d["name"]=="Winter Storm Warning"]
print(len(ws_warn))

ws_watch = d[d["name"]=="Winter Storm Watch"]
print(len(ws_watch))

ww_advisory = d[d["name"]=="Winter Weather Advisory"]
print(len(ww_advisory))

densefog = d[d["name"]=="Dense Fog Advisory"]
print(len(densefog))

group

# Grab events to run for inference 
- Using dataframe read in above
- Overview of flow of prepping data: Must subset to events you want to run/organize in one "launch" of inference jobs. Often, I want to run 3 jobs simulatenously on Hulk. This code below prepares for that level of run. Why does that matter? Because when removing the datetimes that have already had inference runs on, that step uses the "aggregated" dir (/home/csutter/DRIVE-clean/operational_runs/set__aggregate_allruns") which already has consolidate across all the different inference sets, thus the aggregation code needs to have been ran up to that point, so the jobs we're preparing to run here exclude ONLY those that are already DONE running and in aggregation (i.e. you can't start one job for lake effect events, and then prep a new dataset for blizzard events and run that one 3 mins later, bc there may be overlap between the lake effect and blizzard events, and you'd then be running duplicate code.)
- TL;DR - Just make sure you run the aggregated dir code before running this code, and make sure that a job you want to start from this code doesn't rely on/ have duplicate datetimes with another job that is already running. In other words, The d_event_subset below needs to contain the full set from which you will start all your jobs right now.


Events for tracking
- 'Snow Squall Warning' (RAN Dec 2025)
- 'Blizzard Warning' (small set) (RAN Jan 2026)
- 'Lake Effect Snow Warning'
- 'Winter Storm Warning' 
- STILL TO RUN AS OF 1/21/26: 'Winter Storm Watch', 'Dense Fog Advisory', 'Lake Effect Snow Warning', 'Winter Storm Warning', 'Winter Storm Watch', 'Winter Weather Advisory'

In [ ]:
# grab snow squall data (or whatever event of interest is)
d_event_subset = d[d["name"]=='Winter Storm Warning'] # ADJUST HERE!!

# just for reference
print(len(d_event_subset))

In [ ]:
# Grab start and end times, for which we'll run all datetimes in between
# Will grab start time, end time, and list every 5-min increment in between
# Will also run 15 mins before and after squall starts to capture the prior and post conditions
# Round to 0005, 0010, etc, 5 min increments. Why? B/c 1) we only snapshot images in 5-min increments anyway, and while they won't be exactly on the even 5 mins, if we consistently run inference runs with ever 5 mins, it means we'll capture all instances (e.g. if we started allowing "off" times like 11 min rather than 10, that may be the same "image instance" for 1000 cams -- this is a waste of inference run). This allows us to keep track seamlessly across lots of different case study inference runs, exactly which datetimes have and have not been ran already
# Note: we run every instance statewide, just for consistency/ease of run (see note above about seamless tracking across case studies). I'm not parsing certain regions based on those that had the squall in region. 

d_event_subset["start_round"] = d_event_subset["issued"].dt.round("5min") # will want these in a df for reference of the rounded start time
d_event_subset["end_round"] = d_event_subset["expired"].dt.round("5min") # will want these in a df for reference of the rounded ende time

d_event_subset["start_buffer"] = d_event_subset["start_round"] - pd.Timedelta(minutes=15)
d_event_subset["end_buffer"] = d_event_subset["end_round"] + pd.Timedelta(minutes=15)

d_event_subset["all_times"] = d_event_subset.apply(
    lambda row: pd.date_range(start=row["start_buffer"], end=row["end_buffer"], freq="5T"),
    axis=1
)

d_event_subset["all_times_format"] = d_event_subset["all_times"].apply(
    lambda elem: [t.strftime("%Y%m%d_%H%M") for t in elem]
)

d_event_subset.head(4)

In [ ]:
# make a list of all the datetimes we want to run
# nested loop, bc each row (squall instance) has a range of datetimes to run

datetimes_all = []

for i in d_event_subset["all_times_format"]:
    for j in i:
        datetimes_all.append(j)

In [ ]:

dates_list = np.unique(datetimes_all)
print(len(dates_list)) # this is the amount of instances to run!

print(len(datetimes_all))
# Note that some will be overlapping for nearby counties or overlapping zones

print(dates_list[0:4])


# Need to also see which datetimes I've already ran from past inference! See other notebook. 


In [ ]:
# just for investigation, dont need to run

filepath = glob(f"/home/csutter/DRIVE-clean/operational_runs/set__aggregate_allruns/data_6_ensembling/*/*/*/*") # just need the datetime which is in the path name, not all the way down the csv pred level

fs= sorted(filepath)

fs[0:4]

In [ ]:
# remove instances for which we already ran inference on in past studies -- would be duplicate, not needed

# REFERENCE: from this code: /home/csutter/DRIVE-clean/operational_runs/dates_sample.ipynb

# Grab dates already ran
# Note that you will have needed to make sure all runs were aggregated into the agg dir, as that is where this code "looks" for dates already ran. That is done in this file: /home/csutter/DRIVE-clean/operational_analysis/notebooks/analysis.ipynb

from glob import glob

scm_finalpreds = []

filepath = glob(f"/home/csutter/DRIVE-clean/operational_runs/set__aggregate_allruns/data_6_ensembling/*/*/*/*") # just need the datetime which is in the path name, not all the way down the csv pred level

# print(filepath)
# print(len(filepath))

for f in filepath:
    # print(f)
    f = f[-13:]
    scm_finalpreds.append(f)


print("instances for which we have predictions")
print(len(scm_finalpreds))
scm_finalpreds[0]

print(scm_finalpreds[0:3])

# Cross check from dates_list and remove any dates already ran from scm_finalpreds

dates_list_new = []
for d in dates_list:
    if d not in scm_finalpreds:
        dates_list_new.append(d)

print("Unique datetimes")
print(len(dates_list_new))



In [ ]:
print(len(dates_list))
print(len(dates_list_new))  # difference shows removed instances that were already ran

In [ ]:
# Save out 
forcsv = pd.DataFrame(dates_list_new[600:], columns=['date']) # ADJUST HERE!! Manually adjust how many instance to run in one sbatch run. Usually ~400 is good (takes 20 hours)
forcsv

savetodir = "/home/csutter/DRIVE-clean/operational_runs/set28_blizzard3" # ADJUST HERE!! Naming to match what event you're running
os.makedirs(savetodir)
forcsv.to_csv(f"{savetodir}/dates.csv") 

# Grab squall - Events and buffers

### Squall events


In [ ]:
eventsofint = ['Snow Squall Warning']

devents = d[d["name"].isin(eventsofint)]

print(len(devents))
display(devents.head(4))
print(type(devents))

print(np.unique(devents["vtec_year"]))

In [ ]:
print(np.unique([devents["ugc_gis"]])) # NYC is New York County, not city! 

In [ ]:
print(devents.columns)

In [ ]:
print(len(np.unique(devents['eventid'])))



Note on row uniqueness

In [ ]:
print(len(devents[['eventid','ugc_gis','issued']].drop_duplicates())) # Unique by these fields -- IMPORTANT.
# I think what's happening is the eventid resets by year. So we cant just do eventid bc 2022 and 2025 will both have "1", but when we tie in issued, the rows become unique. Important to note that eventid is not unique to location though. eventid is probably closer to "episode" of the ncei database where it's the system
print(len(devents))

In [ ]:
devents[devents["eventid"]==1]

In [ ]:
devents.head(3)

### Prepare data - Don't need to run again! Saved out 
- after running all the events and buffer datetimes as found above

Buffer times

In [ ]:
import pandas as pd
import glob

# The main dataframe made here, df, takes for every row in the squall df - unique by 'eventid','ugc_gis','issued' - and finds the list of buffer times assocaited with that event. This way, we have a list of buffer times CONNECTED to each event

# ==========================================
# 1. GENERATE REQUIRED BUFFER TIMES
# ==========================================

### TO RUN IT ON SQUALL DF
df = devents.copy()
print(f"Original events: {len(df)}")

df['issued'] = pd.to_datetime(df['issued'])
df['expired'] = pd.to_datetime(df['expired'])

df['issued_rounded'] = df['issued'].dt.round('5min')
df['expired_rounded'] = df['expired'].dt.round('5min')

def get_buffer_times(row):
    start = row['issued_rounded']
    end = row['expired_rounded']
    offsets_before = [-60, -45, -30, -15, -10, -5]
    offsets_after = [5, 10, 15, 30, 45, 60]
    
    buffer_list = []
    for offset in offsets_before:
        target_time = start + pd.Timedelta(minutes=offset)
        buffer_list.append(target_time.strftime('%Y%m%d_%H%M'))
    for offset in offsets_after:
        target_time = end + pd.Timedelta(minutes=offset)
        buffer_list.append(target_time.strftime('%Y%m%d_%H%M'))
        
    return buffer_list

df['buffer_datetimes'] = df.apply(get_buffer_times, axis=1)

# --- THE MAGIC TRICK: Explode the lists into individual rows ---
df_exploded = df.explode('buffer_datetimes')

# Rename the column since it's no longer a list of datetimes, just one
df_exploded = df_exploded.rename(columns={'buffer_datetimes': 'buffer_datetime'})

# Slice the dataframe to only keep the mapping columns you need
mapping_cols = ['eventid', 'ugc_gis', 'issued', 'buffer_datetime']
df_mapping = df_exploded[mapping_cols].copy()

# ==========================================
# 2. FIND EXISTING FILES IN YOUR DIRECTORIES
# ==========================================

print("Scanning operational runs for existing files...")

main_dir = "/home/csutter/DRIVE-clean/operational_runs"
search_pattern = f"{main_dir}/*/data_6_ensembling/*/*/*/*/finalpreds.csv"

existing_files = glob.glob(search_pattern)

print(f"Total raw files found on disk (including duplicates): {len(existing_files):,}")

existing_times = set()

for file_path in existing_files:
    time_str = file_path.split('/')[-2]
    existing_times.add(time_str)

# ==========================================
# 3. CROSS-CHECK AND FILTER
# ==========================================

# 1. Filter your mapping DataFrame! 
# Keep only the rows where the buffer_datetime does NOT exist in your on-disk set
df_missing_mapping = df_mapping[~df_mapping['buffer_datetime'].isin(existing_times)]

# 2. Extract the unique, flat list of times you actually need to feed your next script
times_to_run = sorted(df_missing_mapping['buffer_datetime'].unique().tolist())

# --- Summary Output ---
print(f"Total unique buffer times required by dataset: {df_mapping['buffer_datetime'].nunique():,}")
print(f"Total unique timestamps already on disk: {len(existing_times):,}")
print(f"Total unique timestamps left to process: {len(times_to_run):,}")

print("\n--- Your Output Mapping DataFrame (df_missing_mapping) ---")
print(df_missing_mapping.head(10))

# You can now save this mapping DataFrame to use later!
# df_missing_mapping.to_csv("squall_buffer_mapping.csv", index=False)

In [ ]:
display(df_exploded.head(14))
print(len(df_exploded))
print(len(devents))
print(type(df_exploded))

Prepare list of times for running inefence operation runs

In [ ]:
# Print the complete enumerated list of missing times
# THESE are what I run in operataional run sets dates.csv
print("\nComplete list of unique datetimes to run:")
for index, t in enumerate(times_to_run, start=1):
    print(f"{index}, {t}")

Join buffers to the EVENTS df, so they're in one dataframe (needed for analyses)

In [ ]:


# buffer df is this
display(df.head(3)) ## <--- this df is all we need.  This has the list of buffer times ADDED to each event. So this df is all we need to move forward now. Don't need to join to events, since we've already incorporated that in the making of df above. 

# print(df.columns)

# for reference, can see that these dfs are of same length. 
display(devents.head(3)) # squall df without list of buffer times column
# print(devents.columns)
print(len(devents))

print(len(df))


Connect to cams (spatial join)

In [ ]:
#### 2 - Cam lat and lons (not sure we need, just load data for now)

cams = pd.read_csv("/home/csutter/DRIVE/site_analysis/_reference/511NY_API_GetCameras_response.csv")

cams = cams[((cams["Disabled"]==False)&(cams["Blocked"]==False))]
print(len(cams))

In [ ]:
# spatial join cams to EVENTS data

cams_gdf = gpd.GeoDataFrame(
    cams, 
    geometry=gpd.points_from_xy(cams.Longitude, cams.Latitude),
    crs="EPSG:4326" # Start with standard GPS coordinates
)
# print(len(cams_gdf))

cams_gdf = cams_gdf.to_crs(df.crs) # match the CRS to be exactly the system being used in the events dataset
# print(len(cams_gdf))

# Perform the spatial join
events_camloc = gpd.sjoin(
    cams_gdf, 
    df, 
    how="inner",
    predicate="within" # Check if the point is WITHIN the polygon
)

cams_gdf.head(3)

print(len(df[["geometry"]]))
print(len(cams))
print(len(events_camloc)) # Splits an event (which was usually one row) into multiple rows, one per each camera, so they larger df size makes sense

# note that doing the join this way (inner) ensures that we're only keeping events and geometries that have cams in them. 

events_camloc.head(3)

In [ ]:
events_camloc.columns

Expand (pd.explode) each squall event (row) into multiple rows based on time stamp, but a couple of nuances/details:
- There are multiple "types" of timestamps -- during event timestamps, and buffer timestamps - that exist in different columns
- To take care of these, 1) make a new col that has the list of event timestamps, and then expand based on event timestamps in a new column called "timestamp" AND a new column called "timetype" = "event" (this will "keep" the buffer col in there, which is just extra info) and 2) expand based on buffer timestamps, making two new columns named the SAME as in 1) since we will combine them ("timestamp" AND  "timetype" = "buffer")  (which will "keep" the event list, which is just extra info) THEN 3) concat the two dfs which all have the same columns

In [ ]:
df.head(5)
print(df.columns)

In [ ]:
import pandas as pd

# ==========================================
# 0) MAKE ISOLATED COPIES
# ==========================================
df_events_expanded = events_camloc.copy()
df_buffers_expanded = events_camloc.copy()

# ==========================================
# 1) PROCESS df_events_expanded
# ==========================================

# 1a) Generate the list of datetimes during the event
def get_event_times(row):
    # pd.date_range magically grabs every 5 minutes between start and end
    times = pd.date_range(start=row['issued_rounded'], end=row['expired_rounded'], freq='5min')
    # Format them immediately to match your YYYYMMDD_HHMM string format
    return [t.strftime('%Y%m%d_%H%M') for t in times]

df_events_expanded['events_datetimes'] = df_events_expanded.apply(get_event_times, axis=1)

# 1b) Map to 'timestamp', explode, and tag
df_events_expanded['timestamp'] = df_events_expanded['events_datetimes']
df_events_expanded = df_events_expanded.explode('timestamp')
df_events_expanded['timetype'] = "event"

# ==========================================
# 2) PROCESS df_buffers_expanded
# ==========================================

# The list already exists, so pick up at the explode step
df_buffers_expanded['timestamp'] = df_buffers_expanded['buffer_datetimes']
df_buffers_expanded = df_buffers_expanded.explode('timestamp')
df_buffers_expanded['timetype'] = "buffer"

# --- COLUMN ALIGNMENT ---
# Drop the raw list columns from both DataFrames so they match perfectly
# (We check if the column exists first so Pandas doesn't throw an error)
cols_to_drop = ['events_datetimes', 'buffer_datetimes']
df_events_expanded = df_events_expanded.drop(columns=[c for c in cols_to_drop if c in df_events_expanded.columns])
df_buffers_expanded = df_buffers_expanded.drop(columns=[c for c in cols_to_drop if c in df_buffers_expanded.columns])

# ==========================================
# 3) CONCATENATE
# ==========================================

df_final = pd.concat([df_events_expanded, df_buffers_expanded], ignore_index=True)

# Optional: Sort the final DataFrame so your events and buffers flow chronologically per event
df_final = df_final.sort_values(by=['eventid', 'timestamp']).reset_index(drop=True)

# Check your work!
print(f"Total rows in unified dataset: {len(df_final):,}")
df_final = df_final.sort_values(['eventid','ugc_gis','issued','ID'])
display(df_final.head(10)) # can see the cols on the left switch from buffer times to event times




Add in model pred data - RSC

In [ ]:
# takes ~40 sec

# ==========================================
# 1. PREP THE MASTER DATAFRAME
# ==========================================

# Replace hyphens with underscores in the ID column to match the model predictions 'site' column
df_final['ID'] = df_final['ID'].str.replace('-', '_')

# ==========================================
# 2. EXTRACT RELEVANT PREDICTIONS EFFICIENTLY
# ==========================================

base_pred_dir = "/home/csutter/DRIVE-clean/operational_runs_wMRMS/data_6_ensembling"

# The specific columns you want to pull from the model prediction CSVs
cols_to_pull = ['site', 'select', 'select_prob', 'qpe_val', 'precip_flag', 'precip_class','ensembleAvg_dry',
'ensembleAvg_snow',
'ensembleAvg_snow_severe',
'ensembleAvg_wet',
'ensembleAvg_poor_viz', 'img_orig']

all_matched_preds = []
missing_files = []

print("Extracting model predictions... this will take a moment depending on the number of unique timestamps.")

# Group the master dataframe by timestamp. 
# This guarantees we only read each CSV once, even if 50 cameras share the same timestamp.
for ts, group in df_final.groupby('timestamp'):
    
    # Parse the timestamp string (YYYYMMDD_HHMM) to build the folder path
    year = ts[0:4]
    month = ts[4:6]
    day = ts[6:8]
    
    # Construct the exact file path
    file_path = f"{base_pred_dir}/{year}/{month}/{day}/{ts}/finalpreds.csv"
    
    if os.path.exists(file_path):
        # Read the CSV, but ONLY load the columns we actually care about to save RAM
        pred_df = pd.read_csv(file_path, usecols=cols_to_pull)
        
        # Get the list of camera IDs that actually matter for this specific timestamp
        relevant_cameras = group['ID'].unique()
        
        # Filter the prediction dataframe to only include our relevant cameras
        filtered_pred = pred_df[pred_df['site'].isin(relevant_cameras)].copy()
        
        # Stamp it with the timestamp so we can merge it back perfectly later
        filtered_pred['timestamp'] = ts
        
        all_matched_preds.append(filtered_pred)
    else:
        # Keep track of any timestamps that didn't have a corresponding file
        missing_files.append(ts)

# ==========================================
# 3. MERGE PREDICTIONS INTO MASTER DATAFRAME
# ==========================================

if all_matched_preds:
    # Concatenate all the little filtered chunks into one master prediction dataframe
    master_preds_df = pd.concat(all_matched_preds, ignore_index=True)
    
    # Perform a LEFT JOIN. 
    # This keeps every row in df_final, and attaches the prediction data where it matches.
    # If a camera was missing from the CSV, those new columns will safely just be NaN (blank).
    df_final = df_final.merge(
        master_preds_df, 
        left_on=['timestamp', 'ID'], 
        right_on=['timestamp', 'site'], 
        how='left'
    )
    
    # Drop the redundant 'site' column since it's identical to 'ID'
    df_final = df_final.drop(columns=['site'])
    
    print("\nMerge complete!")
    print(f"Total rows in df_final: {len(df_final):,}")
    print(f"Timestamps missing their finalpreds.csv file: {len(missing_files):,}")
else:
    print("\nWARNING: No matching prediction files were found for any timestamps.")

In [ ]:
# Display the first few rows to verify the new columns are attached
print("\nPreview of updated dataframe:")
display(df_final.head(5))

Add in model pred data - ODM
- Adds in the same cols just from the ODM preds. Note that MRMS cols shoudl be the same since those should have just been from the same time!!

In [ ]:
# takes ~40 sec

# ==========================================
# 1. PREP THE MASTER DATAFRAME
# ==========================================

# (Assuming df_final['ID'] already has underscores from your previous run, 
# but leaving this here just in case you are running it fresh)
df_final['ID'] = df_final['ID'].str.replace('-', '_')

# ==========================================
# 2. EXTRACT RELEVANT PREDICTIONS EFFICIENTLY
# ==========================================

base_pred_dir = "/home/csutter/DRIVE-clean/operational_runs_wMRMS/data_odm_3_ensembling"

# The specific columns you want to pull from the model prediction CSVs
cols_to_pull = ['site', 'select', 'select_prob', 'qpe_val', 'precip_flag', 'precip_class']

all_matched_preds = []
missing_files = []

print("Extracting ODM model predictions... this will take a moment.")

for ts, group in df_final.groupby('timestamp'):
    year = ts[0:4]
    month = ts[4:6]
    day = ts[6:8]
    
    file_path = f"{base_pred_dir}/{year}/{month}/{day}/{ts}/finalpreds.csv"
    
    if os.path.exists(file_path):
        pred_df = pd.read_csv(file_path, usecols=cols_to_pull)
        relevant_cameras = group['ID'].unique()
        filtered_pred = pred_df[pred_df['site'].isin(relevant_cameras)].copy()
        filtered_pred['timestamp'] = ts
        
        all_matched_preds.append(filtered_pred)
    else:
        missing_files.append(ts)

# ==========================================
# 3. RENAME AND MERGE PREDICTIONS
# ==========================================

if all_matched_preds:
    master_preds_df = pd.concat(all_matched_preds, ignore_index=True)
    
    # --- THE FIX: Rename the columns before merging ---
    # We map the old names to the new ODM-specific names
    rename_mapping = {
        'select': 'odm_select',
        'select_prob': 'odm_select_prob',
        'qpe_val': 'odm_qpe_val',
        'precip_flag': 'odm_precip_flag',
        'precip_class': 'odm_precip_class'
    }
    master_preds_df = master_preds_df.rename(columns=rename_mapping)
    
    # Perform the LEFT JOIN
    df_final = df_final.merge(
        master_preds_df, 
        left_on=['timestamp', 'ID'], 
        right_on=['timestamp', 'site'], 
        how='left'
    )
    
    # Drop the redundant 'site' column
    df_final = df_final.drop(columns=['site'])
    
    print("\nODM Merge complete!")
    print(f"Total rows in df_final: {len(df_final):,}")
    print(f"Timestamps missing their ODM finalpreds.csv file: {len(missing_files):,}")
else:
    print("\nWARNING: No matching ODM prediction files were found for any timestamps.")



In [ ]:
print(len(df_final))

Add in CNN-only pred

In [ ]:
import os
import glob
import pandas as pd

# ==========================================
# 1. PREP THE MASTER DATAFRAME
# ==========================================

df_final['ID'] = df_final['ID'].str.replace('-', '_')

# ==========================================
# 2. EXTRACT AND ENSEMBLE CNN PREDICTIONS
# ==========================================

base_dir = "/home/csutter/DRIVE-clean/operational_runs"

# The specific columns you want to pull from the 5 model CSVs
cols_to_pull = [
    'img_orig', 'site', 'dt_str',
    'calib_prob_dry', 'calib_prob_poor_viz', 'calib_prob_snow', 
    'calib_prob_snow_severe', 'calib_prob_wet'
]

all_matched_preds = []
missing_files = []

print("Extracting and ensembling CNN predictions... this will take a moment.")

for ts, group in df_final.groupby('timestamp'):
    year = ts[0:4]
    month = ts[4:6]
    day = ts[6:8]
    
    # Use glob to find the directory across any 'set' folder
    # This will return a list of all paths that match this pattern
    search_pattern = f"{base_dir}/*/data_3_cnncalib/{year}/{month}/{day}/{ts}"
    matching_dirs = glob.glob(search_pattern)
    
    if matching_dirs:
        # Take the first matched directory (ignores duplicates in other sets)
        target_dir = matching_dirs[0]
        
        # Read and collect the 5 models (m0 through m4)
        model_dfs = []
        for m in range(5):
            file_path = f"{target_dir}/cnncalib_m{m}.csv"
            if os.path.exists(file_path):
                # We use try/except just in case one of the CSVs is corrupted or missing columns
                try:
                    df_m = pd.read_csv(file_path, usecols=cols_to_pull)
                    model_dfs.append(df_m)
                except Exception as e:
                    print(f"Warning: Could not read {file_path}. Error: {e}")
        
        # If we successfully loaded the models, ensemble them
        if model_dfs:
            # Combine all models into one tall dataframe
            combined_models = pd.concat(model_dfs, ignore_index=True)
            
            # Filter down to only the cameras relevant to this timestamp to save processing time
            relevant_cameras = group['ID'].unique()
            combined_models = combined_models[combined_models['site'].isin(relevant_cameras)]
            
            if not combined_models.empty:
                # Group by the static columns and take the mean of the probability columns
                static_cols = ['img_orig', 'site','dt_str']
                ensemble_df = combined_models.groupby(static_cols, as_index=False).mean()
                
                # Rename the probability columns to the requested 'cnn_' format
                rename_mapping = {
                    'calib_prob_dry': 'cnn_dry',
                    'calib_prob_poor_viz': 'cnn_poor_viz',
                    'calib_prob_snow': 'cnn_snow',
                    'calib_prob_snow_severe': 'cnn_snow_severe',
                    'calib_prob_wet': 'cnn_wet'
                }
                ensemble_df = ensemble_df.rename(columns=rename_mapping)
                
                # Determine cnn_select and cnn_select_prob
                prob_cols = list(rename_mapping.values())
                
                # idxmax(axis=1) returns the column name with the highest value (e.g., 'cnn_dry')
                # We strip out 'cnn_' to just get the class name (e.g., 'dry')
                ensemble_df['cnn_select'] = ensemble_df[prob_cols].idxmax(axis=1).str.replace('cnn_', '')
                
                # max(axis=1) returns the actual highest numerical probability
                ensemble_df['cnn_select_prob'] = ensemble_df[prob_cols].max(axis=1)
                
                # Add the timestamp back so we can merge it
                ensemble_df['timestamp'] = ts
                
                all_matched_preds.append(ensemble_df)
        else:
            # The directory existed, but none of the m0-m4 files were readable
            missing_files.append(ts)
            
    else:
        # Glob found 0 matching directories
        missing_files.append(ts)

# ==========================================
# 3. MERGE PREDICTIONS INTO DF_FINAL
# ==========================================

if all_matched_preds:
    master_preds_df = pd.concat(all_matched_preds, ignore_index=True)
    
    master_preds_df = master_preds_df.drop(columns=['img_orig'])
    # Perform the LEFT JOIN
    df_final = df_final.merge(
        master_preds_df, 
        left_on=['timestamp', 'ID'], 
        right_on=['timestamp', 'site'], 
        how='left'
    )
    
    # Drop the redundant 'site' column from the right side of the merge
    df_final = df_final.drop(columns=['site'])
    
    print("\nCNN Ensemble & Merge complete!")
    print(f"Total rows in df_final: {len(df_final):,}")
    print(f"Timestamps completely missing data directories: {len(missing_files):,}")
else:
    print("\nWARNING: No matching CNN prediction directories were found for any timestamps.")

In [ ]:
df_final.to_csv("/home/csutter/DRIVE-clean/weather_events/data/squall_casestudies/all_squalldata_and_modelpreds/all_squalldata_and_modelpreds.csv")

In [ ]:
print(len(df_final))

In [ ]:
df_final.head(4)

In [ ]:
# Display the first few rows to verify the new columns are attached
print("\nPreview of updated dataframe:")
# display(df_final[50:70])

### Squall analysis

In [ ]:
# Read in data saved above

df_final = pd.read_csv("/home/csutter/DRIVE-clean/weather_events/data/squall_casestudies/all_squalldata_and_modelpreds/all_squalldata_and_modelpreds.csv")

display(df_final.head(4))

In [ ]:
print(df_final.columns)

In [ ]:
df_final.head(4)

In [ ]:
# First find "events" that have a large number of cameras
# See note on row uniqueness farther up

In [ ]:
# Find events that have large camera coverage 
# need that to be able to make the argument / contribution being posed in this work

In [ ]:
print(np.unique(df_final["ugc_gis"]))

In [ ]:
# 1. Isolate the core events
df_events_only = df_final[df_final['timetype'] == 'event'].copy()

# 2. Aggregate the statistics, now including 'loc_desc'
# (Assuming loc_desc made it into your df_final. If it didn't, let me know and we can merge it!)
coverage_stats = df_events_only.groupby(
    ['eventid', 'ugc_gis', 'loc_desc', 'issued']
).agg(
    total_unique_cameras=('ID', 'nunique'),
    total_data_points=('timestamp', 'count')
).reset_index()

# 3. Sort to find the most data-rich events
top_coverage_events = coverage_stats.sort_values(
    by=['total_unique_cameras', 'total_data_points'], 
    ascending=[False, False]
)


In [ ]:
df_events_only.head(3)

In [ ]:
display(top_coverage_events.head(30))

# let's try eventid = 1, from 2025-02-7, for Monroe (Rochester), Eerie (Buffalo) (rows 21 and 5)

# For a different one, eventid = 4 from 2025-01-28,  Onondaga (row 165)

# Can try a city one too?

# NOTE Monroe, 2025-02-07, events 1 2 and 3, all are the same system....  see super event code below. 173 cams so it's a good event
# 1, NYC055, 2025-02-07 06:25:00 
# 2, NYC055, 2025-02-07 07:15:00 
# 3, NYC055, 2025-02-07 08:17:00	
# Gemini: Because Snow Squall Warnings (SQWs) are short-fused (usually only valid for 30 to 45 minutes), the NWS cannot just issue a 3-hour warning for a squall. If a squall is moving slowly, stalls, or if multiple lake-effect bands are tracking over the exact same area back-to-back, the NWS will issue a warning, let it expire, and immediately issue a new one (or issue a new one that slightly overlaps the old one geographically).

In [ ]:
# CLEAN DATA! MUST RUN

df_final = df_final.dropna(subset=['select', 'odm_select']).copy() # ADDED TO REMOVE NAN ROWS, WHICH ARE ROWS WITHOUT MODEL PREDS

print(len(df_final))

# Since there are close squall warnings, one time may be a buffer time for warning 1 while it may actually fall under the warning for warning 2. Keep the row that is the warning, timetype = event, rather than the duplicate timetype = buffer row

# # 1. Get a list of all columns EXCEPT 'timetype'
# subset_cols = [col for col in df_final.columns if col != 'timetype']

# # 2. Sort the dataframe. 
# # Alphabetically, 'event' comes after 'buffer'. 
# # By sorting descending (ascending=False), 'event' gets pushed to the top.
# df_sorted = df_final.sort_values(by='timetype', ascending=False)

# # 3. Drop the duplicates based on the subset columns, keeping the first one it sees
# df_final = df_sorted.drop_duplicates(subset=subset_cols, keep='first')

# print(len(df_final))

# import pandas as pd

# # 1. Get a list of all columns EXCEPT 'timetype'
# subset_cols = [col for col in df_final.columns if col != 'timetype']

# # 2. Sort the dataframe. 
# # Alphabetically, 'event' comes after 'buffer'. 
# # By sorting descending (ascending=False), 'event' gets pushed to the top.
# df_sorted = df_final.sort_values(by='timetype', ascending=False)

# # 3. Drop the duplicates based on the subset columns, keeping the first one it sees
# df_clean = df_sorted.drop_duplicates(subset=subset_cols, keep='first')

# print(len(df_clean))


# # 1. Sort the dataframe so 'event' (alphabetically later) is above 'buffer'
# df_sorted = df_final.sort_values(by='timetype', ascending=False)

# # 2. Drop duplicates based ONLY on the specific camera and the timestamp
# # Because we sorted 'event' to the top, keep='first' automatically keeps 'event' and drops 'buffer'
# df_final = df_sorted.drop_duplicates(subset=['ID', 'timestamp'], keep='first')

# # Check your results
# print(f"Original length: {len(df_final)}")
# print(f"Cleaned length: {len(df_final)}")


# 1. Create a strict numerical priority column: 'event' gets #1, everything else gets #2
df_final['sort_priority'] = np.where(df_final['timetype'] == 'event', 1, 2)

# 2. Sort by ID, timestamp, AND our new priority (ascending=True so 1 floats to the top)
df_sorted = df_final.sort_values(by=['ID', 'timestamp', 'sort_priority'], ascending=True)

# 3. Drop duplicates. Because priority 1 is at the top, 'event' will ALWAYS win.
df_final = df_sorted.drop_duplicates(subset=['ID', 'timestamp'], keep='first')

# (Optional) Clean up the temporary priority column
df_final = df_final.drop(columns=['sort_priority'])

# Check your results
print(f"Cleaned length: {len(df_final)}")


In [ ]:
df_final[((df_final["ID"]=="Skyline_1811")&(df_final["timestamp"]=="20250207_0700"))]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Isolate the specific Monroe County event
target_event_id = 1  # Make sure this matches the data type in your df (int or string)
target_ugc = "NYC055"
issued = "2025-02-07 06:25:00"

# # Monroe, not great result, weird
# target_event_id = 1  # Make sure this matches the data type in your df (int or string)
# target_ugc = "NYC055"
# issued = "2025-02-07 06:25:00"

# Eerie, also kind of strange with dip downs of non-obs
# target_event_id = 1  # Make sure this matches the data type in your df (int or string)
# target_ugc = "NYC029"
# issued = "2025-02-07 06:25:00"

# Onandaga, Not super meaningful 
# target_event_id = 4  # Make sure this matches the data type in your df (int or string)
# target_ugc = "NYC067"
# issued = "2025-01-28 09:24:00"

# this one is OKAY
# target_event_id = 7  # Make sure this matches the data type in your df (int or string)
# target_ugc = "NYC103"
# issued = "2022-02-19 20:59:00"

# Filter df_final for just this event and its buffers
event_df = df_final[(df_final['eventid'] == target_event_id) & (df_final['ugc_gis'] == target_ugc)& (df_final['issued'] == issued)].copy()

event_df

# Ensure timestamp is a proper datetime object for plotting
event_df['datetime'] = pd.to_datetime(event_df['timestamp'], format='%Y%m%d_%H%M')

# Sort chronologically
event_df = event_df.sort_values('datetime')

# Define what a "Squall Signature" is (Update these strings to match your exact model outputs!)
obs_class = 'obs'
rsc_class = ['snow_severe','poor_viz']

# Create the binary hit flag
event_df['squall_hit'] = ((event_df['odm_select'] == obs_class) | (event_df['select'].isin(rsc_class))).astype(int)

display(event_df.head(4))

display(event_df[event_df["select"]=="poor_viz"].head(3))

In [ ]:
# 1. Create specific binary flags BEFORE aggregating
# (Adjust the string names to exactly match your model's output classes)
snow_class = 'snow_severe'
poor_viz_class = 'poor_viz' 
obs_class = 'obs'

super_event_df['hit_severe_snow'] = (super_event_df['select'] == snow_class).astype(int)
super_event_df['hit_poor_viz'] = (super_event_df['select'] == poor_viz_class).astype(int)
super_event_df['hit_odm_obs'] = (super_event_df['odm_select'] == obs_class).astype(int)

# 2. Aggregate data by time, now including our new columns
temporal_stats = super_event_df.groupby(['datetime', 'timetype']).agg(
    total_active_cams=('ID', 'count'),
    total_hits=('squall_hit', 'sum'),
    total_severe_snow=('hit_severe_snow', 'sum'),
    total_poor_viz=('hit_poor_viz', 'sum'),
    total_odm_obs=('hit_odm_obs', 'sum')
).reset_index()

# 3. Calculate the percentages for all categories
temporal_stats['percent_impacted'] = (temporal_stats['total_hits'] / temporal_stats['total_active_cams']) * 100
temporal_stats['percent_severe_snow'] = (temporal_stats['total_severe_snow'] / temporal_stats['total_active_cams']) * 100
temporal_stats['percent_poor_viz'] = (temporal_stats['total_poor_viz'] / temporal_stats['total_active_cams']) * 100
temporal_stats['percent_odm_obs'] = (temporal_stats['total_odm_obs'] / temporal_stats['total_active_cams']) * 100

# 4. Plotting
plt.figure(figsize=(14, 6))

# Plot the original combined "Master" line (made thicker/dashed to stand out)
# sns.lineplot(data=temporal_stats, x='datetime', y='percent_impacted', 
#              linewidth=3, color='black', linestyle='--', label='Combined Squall Hit (Any)')

# Plot the 3 specific class lines
sns.lineplot(data=temporal_stats, x='datetime', y='percent_severe_snow', 
             linewidth=2, color='red', marker = 'o', label=f'RSC: {snow_class}')
sns.lineplot(data=temporal_stats, x='datetime', y='percent_poor_viz', 
             linewidth=2, color='gray', marker = 'o', label=f'RSC: {poor_viz_class}')
sns.lineplot(data=temporal_stats, x='datetime', y='percent_odm_obs', 
             linewidth=2, color='black', marker = 'o',  linestyle='--', label=f'ODM: {obs_class}')

# Visually separate the "Buffer" times from the "Event" times
event_start = temporal_stats[temporal_stats['timetype'] == 'event']['datetime'].min()
event_end = temporal_stats[temporal_stats['timetype'] == 'event']['datetime'].max()
# plt.axvspan(event_start, event_end, color='red', alpha=0.1, label='NWS Warning Active')

# Draw a red box for each of the warning issuances in this time window
unique_events = super_event_df[super_event_df['timetype'] == 'event']['eventid'].unique()

# Uncomment this to show the Squall warning shading
for eid in unique_events:
    # Find the start and end of this specific eventid
    evt_data = super_event_df[(super_event_df['eventid'] == eid) & (super_event_df['timetype'] == 'event')]
    if not evt_data.empty:
        e_start = evt_data['datetime'].min()
        e_end = evt_data['datetime'].max()
        plt.axvspan(e_start, e_end, color='red', alpha=0.15, label=f'NWS Squall Warning' if eid == unique_events[0] else "")
        
plt.title(f"2025 Feb 7 (Snow Squall Warning) in Albany County - Model Predictions", fontsize=20)
plt.xlabel("Time (UTC)", fontsize=16)
plt.ylabel("Percentage of Cameras (%)", fontsize=16)
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
# plt.savefig("monroe_hazard_footprint_detailed.png")



In [ ]:
# 1. Create specific binary flags BEFORE aggregating
# (Adjust the string names to exactly match your model's output classes)
snow_class = 'snow_severe'
poor_viz_class = 'poor_viz' 
obs_class = 'obs'

event_df['hit_severe_snow'] = (event_df['select'] == snow_class).astype(int)
event_df['hit_poor_viz'] = (event_df['select'] == poor_viz_class).astype(int)
event_df['hit_odm_obs'] = (event_df['odm_select'] == obs_class).astype(int)

# 2. Aggregate data by time, now including our new columns
temporal_stats = event_df.groupby(['datetime', 'timetype']).agg(
    total_active_cams=('ID', 'count'),
    total_hits=('squall_hit', 'sum'),
    total_severe_snow=('hit_severe_snow', 'sum'),
    total_poor_viz=('hit_poor_viz', 'sum'),
    total_odm_obs=('hit_odm_obs', 'sum')
).reset_index()

# 3. Calculate the percentages for all categories
temporal_stats['percent_impacted'] = (temporal_stats['total_hits'] / temporal_stats['total_active_cams']) * 100
temporal_stats['percent_severe_snow'] = (temporal_stats['total_severe_snow'] / temporal_stats['total_active_cams']) * 100
temporal_stats['percent_poor_viz'] = (temporal_stats['total_poor_viz'] / temporal_stats['total_active_cams']) * 100
temporal_stats['percent_odm_obs'] = (temporal_stats['total_odm_obs'] / temporal_stats['total_active_cams']) * 100

# 4. Plotting
plt.figure(figsize=(14, 6))

# Plot the original combined "Master" line (made thicker/dashed to stand out)
sns.lineplot(data=temporal_stats, x='datetime', y='percent_impacted', 
             linewidth=3, color='black', linestyle='--', label='Combined Squall Hit (Any)')

# Plot the 3 specific class lines
sns.lineplot(data=temporal_stats, x='datetime', y='percent_severe_snow', 
             linewidth=2, color='blue', marker = 'o', label=f'CNN: {snow_class}')
sns.lineplot(data=temporal_stats, x='datetime', y='percent_poor_viz', 
             linewidth=2, color='cyan', marker = 'o', label=f'CNN: {poor_viz_class}')
sns.lineplot(data=temporal_stats, x='datetime', y='percent_odm_obs', 
             linewidth=2, color='orange', marker = 'o',  label=f'ODM: {obs_class}')

# Visually separate the "Buffer" times from the "Event" times
event_start = temporal_stats[temporal_stats['timetype'] == 'event']['datetime'].min()
event_end = temporal_stats[temporal_stats['timetype'] == 'event']['datetime'].max()
plt.axvspan(event_start, event_end, color='red', alpha=0.1, label='NWS Warning Active')

plt.title(f"Sub-Polygon Hazard Dynamics: Event {target_event_id}", fontsize=14)
plt.xlabel("Time (UTC)")
plt.ylabel("Percentage of Cameras (%)")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
# plt.savefig("monroe_hazard_footprint_detailed.png")

In [ ]:
event_df[event_df["odm_select"]=="obs"][130:140]

In [ ]:
event_df.groupby(["timestamp","odm_select"]).count()

In [ ]:
event_df[(event_df["odm_select"]=="obs")&(event_df["timestamp"]=="20250207_0730")]

In [ ]:
event_df[(event_df["odm_select"]=="nonobs")&(event_df["timestamp"]=="20250207_0700")][20:26]

In [ ]:
event_df[event_df["ID"]=="Skyline_5046"]

In [ ]:
event_df[event_df["ID"]=="Skyline_5012"]

In [ ]:
event_df[event_df["ID"]=="Skyline_1805"]

In [ ]:
# Buffalo example to use: Skyline_5046, 2/7/25 ~730


super event Monroe county -- see how the 2025-02-07 event from monroe has eventid 1 2 and 3 all back to back? need to piecmeal them together

In [ ]:


# 1. Define the parameters for your "Super Event"
target_ugc = "NYC055"

# We know the first event issued at 06:25 and the last at 08:17. 
# Let's grab everything from 05:00 to 10:00 to capture all buffers and the full decay.
start_time = "2025-02-07 05:00:00"
end_time = "2025-02-07 10:00:00"

# 2. Convert df_final timestamps to actual datetime objects if they aren't already
if not pd.api.types.is_datetime64_any_dtype(df_final['timestamp']):
    df_final['datetime'] = pd.to_datetime(df_final['timestamp'], format='%Y%m%d_%H%M')

# 3. Slice the dataframe using time and location, ignoring eventid
super_event_df = df_final[
    (df_final['ugc_gis'] == target_ugc) & 
    (df_final['datetime'] >= pd.to_datetime(start_time)) &
    (df_final['datetime'] <= pd.to_datetime(end_time))
].copy()

# Sort it chronologically just to be safe
super_event_df = super_event_df.sort_values('datetime')

# 4. Rebuild the Squall Hit logic for this new isolated dataframe
obs_class = 'obs'
snow_class = 'snow_severe'
poor_viz_class = 'poor_viz' 

super_event_df['hit_severe_snow'] = (super_event_df['select'] == snow_class).astype(int)
super_event_df['hit_poor_viz'] = (super_event_df['select'] == poor_viz_class).astype(int)
super_event_df['hit_odm_obs'] = (super_event_df['odm_select'] == obs_class).astype(int)

# Master squall hit flag
super_event_df['squall_hit'] = (super_event_df['hit_severe_snow'] | super_event_df['hit_odm_obs'])

print(f"Super Event extracted! Total 5-minute camera snapshots: {len(super_event_df):,}")

super_event_df.head(5)

In [ ]:
# FOR REFERENCE ONLY - see the super event from ORIGINAL devents df (need to read in, first cell of notebook) to see the overlapping times of warnings

# 1, NYC055, 2025-02-07 06:25:00 
# 2, NYC055, 2025-02-07 07:15:00 
# 3, NYC055, 2025-02-07 08:17:00

eventsofint = ['Snow Squall Warning']

devents = d[d["name"].isin(eventsofint)]

se1 = devents[(devents['eventid'] == 1) & (devents['ugc_gis'] == "NYC055")& (devents['issued'] == "2025-02-07 06:25:00")]

se2 = devents[(devents['eventid'] == 2) & (devents['ugc_gis'] == "NYC055")& (devents['issued'] == "2025-02-07 07:15:00")]

se3 = devents[(devents['eventid'] == 3) & (devents['ugc_gis'] == "NYC055")& (devents['issued'] == "2025-02-07 08:17:00")]

display(se1.head(3))
display(se2.head(3))
display(se3.head(3))

# can see there is an overlap in warning, which is why there is shaded over red portion

In [ ]:
super_event_df.head(5)

# super_event_during = super_event_df[super_event_df["timetype"]=="event"]

In [ ]:
# 1. Create specific binary flags BEFORE aggregating
# (Adjust the string names to exactly match your model's output classes)
snow_class = 'snow_severe'
poor_viz_class = 'poor_viz' 
obs_class = 'obs'

super_event_df['hit_severe_snow'] = (super_event_df['select'] == snow_class).astype(int)
super_event_df['hit_poor_viz'] = (super_event_df['select'] == poor_viz_class).astype(int)
super_event_df['hit_odm_obs'] = (super_event_df['odm_select'] == obs_class).astype(int)

# 2. Aggregate data by time, now including our new columns
temporal_stats = super_event_df.groupby(['datetime', 'timetype']).agg(
    total_active_cams=('ID', 'count'),
    total_hits=('squall_hit', 'sum'),
    total_severe_snow=('hit_severe_snow', 'sum'),
    total_poor_viz=('hit_poor_viz', 'sum'),
    total_odm_obs=('hit_odm_obs', 'sum')
).reset_index()

# 3. Calculate the percentages for all categories
temporal_stats['percent_impacted'] = (temporal_stats['total_hits'] / temporal_stats['total_active_cams']) * 100
temporal_stats['percent_severe_snow'] = (temporal_stats['total_severe_snow'] / temporal_stats['total_active_cams']) * 100
temporal_stats['percent_poor_viz'] = (temporal_stats['total_poor_viz'] / temporal_stats['total_active_cams']) * 100
temporal_stats['percent_odm_obs'] = (temporal_stats['total_odm_obs'] / temporal_stats['total_active_cams']) * 100

# 4. Plotting
plt.figure(figsize=(14, 6))

# Plot the original combined "Master" line (made thicker/dashed to stand out)
# sns.lineplot(data=temporal_stats, x='datetime', y='percent_impacted', 
#              linewidth=3, color='black', linestyle='--', label='Combined Squall Hit (Any)')

# Plot the 3 specific class lines
sns.lineplot(data=temporal_stats, x='datetime', y='percent_severe_snow', 
             linewidth=2, color='red', marker = 'o', label=f'RSC: {snow_class}')
sns.lineplot(data=temporal_stats, x='datetime', y='percent_poor_viz', 
             linewidth=2, color='gray', marker = 'o', label=f'RSC: {poor_viz_class}')
sns.lineplot(data=temporal_stats, x='datetime', y='percent_odm_obs', 
             linewidth=2, color='black', marker = 'o',  linestyle='--', label=f'ODM: {obs_class}')

# Visually separate the "Buffer" times from the "Event" times
event_start = temporal_stats[temporal_stats['timetype'] == 'event']['datetime'].min()
event_end = temporal_stats[temporal_stats['timetype'] == 'event']['datetime'].max()
# plt.axvspan(event_start, event_end, color='red', alpha=0.1, label='NWS Warning Active')

# # Draw a red box for each of the warning issuances in this time window
# unique_events = super_event_df[super_event_df['timetype'] == 'event']['eventid'].unique()

# # Uncomment this to show the Squall warning shading
# for eid in unique_events:
#     # Find the start and end of this specific eventid
#     evt_data = super_event_df[(super_event_df['eventid'] == eid) & (super_event_df['timetype'] == 'event')]
#     if not evt_data.empty:
#         e_start = evt_data['datetime'].min()
#         e_end = evt_data['datetime'].max()
#         plt.axvspan(e_start, e_end, color='red', alpha=0.15, label=f'NWS Squall Warning' if eid == unique_events[0] else "")

# # 1. Extract the absolute NWS warning times directly from the dataframe
# true_warnings = super_event_df[['iso_issued', 'iso_expired']].drop_duplicates().dropna()

# # 2. Loop through them and draw the background boxes
# for _, row in true_warnings.iterrows():
#     # Convert the ISO strings to pandas datetimes so they map to the X-axis properly
#     issue_time = pd.to_datetime(row['iso_issued'])
#     expire_time = pd.to_datetime(row['iso_expired'])
    
#     # Plot the red warning block
#     plt.axvspan(issue_time, expire_time, color='red', alpha=0.1, label='NWS Squall Warning')

# # (Optional: To prevent 'NWS Squall Warning' from appearing 3 times in your legend, 
# # you can handle duplicate legend labels using standard matplotlib deduplication)
# handles, labels = plt.gca().get_legend_handles_labels()
# by_label = dict(zip(labels, handles))
# plt.legend(by_label.values(), by_label.keys())

        
# plt.title(f"2025 Feb 7 (Snow Squall Warning) in Monroe County - Model Predictions", fontsize=20)
# plt.xlabel("Time (UTC)", fontsize=16)
# plt.ylabel("Percentage of Cameras (%)", fontsize=16)
# plt.legend(fontsize=12)
# plt.grid(True, linestyle='--', alpha=0.6)
# plt.tight_layout()
# # plt.savefig("monroe_hazard_footprint_detailed.png")

# 1. Extract the absolute NWS warning times directly from the dataframe
true_warnings = super_event_df[['iso_issued', 'iso_expired']].drop_duplicates().dropna()

# 2. Loop through them and draw the background boxes
for _, row in true_warnings.iterrows():
    # Convert the ISO strings to pandas datetimes so they map to the X-axis properly
    issue_time = pd.to_datetime(row['iso_issued'])
    expire_time = pd.to_datetime(row['iso_expired'])
    
    # Plot the red warning block
    plt.axvspan(issue_time, expire_time, color='red', alpha=0.1, label='NWS Squall Warnings')

plt.title("2025 Feb 7 (Snow Squall Warnings) in Monroe County - Model Predictions", fontsize=20)
plt.xlabel("Time (UTC)", fontsize=16)
plt.ylabel("Percentage of Cameras (%)", fontsize=16)

# ==========================================
# CUSTOM LEGEND DEDUPLICATION
# ==========================================
# Grab all the labels, zip them into a dictionary (which automatically overwrites duplicates),
# and build the final legend with the fontsize included here!
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys(), fontsize=12)

plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
# plt.savefig("monroe_hazard_footprint_detailed.png")



In [ ]:
super_event_df.head(3)

In [ ]:
print(np.unique(super_event_df["iso_issued"]))
print(np.unique(super_event_df["iso_expired"]))



In [ ]:
# to save out - dont need to run again
super_event_df.to_csv("/home/csutter/DRIVE-clean/weather_events/data/squall_casestudies/monroe/squall20250207_preds.csv")

In [ ]:
super_event_df[((super_event_df["ID"]=="Skyline_1811")&(super_event_df["datetime"]=="2025-02-07 07:35:00"))]

In [ ]:
super_event_df[((super_event_df["select"]=="snow_severe")&(super_event_df["datetime"]=="2025-02-07 07:30:00"))]

In [ ]:
super_event_df[super_event_df["ID"]=="Skyline_1811"]

In [ ]:
print(temporal_stats[['datetime', 'total_active_cams', 'total_hits']].head(20))

In [ ]:
# the dips ^ could just be the storm moving out of an area that has camera coverage

In [ ]:
super_event_df.columns

In [ ]:
print(type(super_event_df))

super_event_df.head(3)

In [ ]:

# 1. Rename the column (Just once!)
# We use errors='ignore' so it doesn't crash if you run the cell twice
super_event_df = super_event_df.rename(columns={"geometry": "geometry_campoint"}, errors="ignore")

# 2. Convert text to spatial objects
# We add a quick check to make sure we only apply wkt.loads if it's still text
if isinstance(super_event_df['geometry_campoint'].iloc[0], str):
    super_event_df['geometry_campoint'] = super_event_df['geometry_campoint'].apply(wkt.loads)

# 3. THE MISSING LINK: Promote it from a regular Pandas DataFrame to a GeoDataFrame!
super_event_df = gpd.GeoDataFrame(super_event_df, geometry="geometry_campoint")

# 4. Load the relevant County shapefile safely
county_shape = devents[devents['ugc_gis'] == target_ugc].iloc[0:1]

# 5. Test the plot!

# 1. First, explicitly tell GeoPandas that our current numbers are standard GPS Lat/Lon (EPSG:4326)
if super_event_df.crs is None:
    super_event_df = super_event_df.set_crs("EPSG:4326")
if county_shape.crs is None:
    county_shape = county_shape.set_crs("EPSG:4326")

# 2. Re-project both dataframes to Web Mercator (EPSG:3857) to fix the stretching
super_event_df = super_event_df.to_crs("EPSG:3857")
county_shape = county_shape.to_crs("EPSG:3857")

ax = county_shape.plot(color='white', edgecolor='black', figsize=(8, 8))
super_event_df.plot(ax=ax, color='lightgrey', markersize=10)

In [ ]:
import pandas as pd

# Isolate the core action time (e.g., 07:15 to 08:30)
core_times_df = super_event_df[(super_event_df['datetime'] >= '2025-02-07 07:00:00') & 
                          (super_event_df['datetime'] <= '2025-02-07 08:30:00')]

# Grab 6 evenly spaced timestamps to plot
unique_times = sorted(core_times_df['datetime'].unique())
times_to_plot = unique_times[::max(1, len(unique_times)//6)][:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, ts in enumerate(times_to_plot):
    ax = axes[i]
    
    # 1. Plot the county outline
    county_shape.plot(ax=ax, color='white', edgecolor='black')
    
    # 2. Plot ALL cameras at this timestamp as small grey dots (shows coverage)
    time_data = super_event_df[super_event_df['datetime'] == ts]
    time_data.plot(ax=ax, color='lightgrey', markersize=20, label='No Squall')
    
    # 3. Plot the SQUALL HITS as large red dots
    hits = time_data[time_data['squall_hit'] == 1]
    if not hits.empty:
        hits.plot(ax=ax, color='red', markersize=60, edgecolor='black', label='Squall Hit')
    
    # --- THE FIX IS HERE ---
    # Convert the numpy datetime back to a pandas timestamp so strftime works
    formatted_time = pd.to_datetime(ts).strftime('%H:%M')
    ax.set_title(f"Time: {formatted_time} UTC", fontsize=12)
    ax.axis('off')

# Add a master legend to the first plot
axes[0].legend(loc='upper left')
plt.suptitle("Spatial Progression of Snow Squall - Monroe County", fontsize=16)
plt.tight_layout()
# plt.savefig("squall_progression_grid.png")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

# Isolate the core action time
core_times_df = super_event_df[(super_event_df['datetime'] >= '2025-02-07 07:00:00') & 
                          (super_event_df['datetime'] <= '2025-02-07 10:00:00')]

# Grab 6 evenly spaced timestamps to plot
# unique_times = sorted(core_times_df['datetime'].unique())
# times_to_plot = unique_times[::max(1, len(unique_times)//15)][:15]

# 1. Get the sorted list of all available times
unique_times = sorted(core_times_df['datetime'].unique())

# 2. Mathematically calculate 15 evenly spaced indices from start to finish
indices = np.linspace(0, len(unique_times) - 1, 16).astype(int)

# 3. Pull those specific times
times_to_plot = [unique_times[i] for i in indices]

fig, axes = plt.subplots(4, 4, figsize=(24, 14))
axes = axes.flatten()

for i, ts in enumerate(times_to_plot):
    ax = axes[i]
    
    # 1. Plot the county outline
    county_shape.plot(ax=ax, color='white', edgecolor='black')
    
    # Extract data for this exact timestamp
    time_data = super_event_df[super_event_df['datetime'] == ts]
    
    # 2. Plot BASE cameras (No Severe Snow and No Poor Viz)
    base_cams = time_data[(time_data['hit_severe_snow'] == 0) & (time_data['hit_poor_viz'] == 0)]
    if not base_cams.empty:
        base_cams.plot(ax=ax, color='whitesmoke', edgecolor='none', markersize=20)
    
    # 3. Plot POOR VIZ hits (Excluding Severe Snow so they don't plot twice)
    pv_cams = time_data[(time_data['hit_poor_viz'] == 1) & (time_data['hit_severe_snow'] == 0)]
    if not pv_cams.empty:
        # Added a black edge color so the darkgray dot pops against the white map
        pv_cams.plot(ax=ax, color='darkgray', edgecolor='black', markersize=60)
        
    # 4. Plot SEVERE SNOW hits (Takes top priority)
    ss_cams = time_data[time_data['hit_severe_snow'] == 1]
    if not ss_cams.empty:
        ss_cams.plot(ax=ax, color='red', edgecolor='black', markersize=60)
    
    # Format time
    formatted_time = pd.to_datetime(ts).strftime('%H:%M')
    ax.set_title(f"Time: {formatted_time} UTC", fontsize=12)
    ax.axis('off')

# ==========================================
# CUSTOM LEGEND CONSTRUCTION
# ==========================================
# Cleaned up to only show the CNN outputs
legend_elements = [
    mlines.Line2D([0], [0], marker='o', color='w', markerfacecolor='black', markersize=8, label='No hazard'),
    mlines.Line2D([0], [0], marker='o', color='w', markerfacecolor='darkgray', markeredgecolor='black', markersize=12, label='CNN: Poor Viz'),
    mlines.Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markeredgecolor='black', markersize=12, label='CNN: Severe Snow')
]

# Add the custom legend to the first plot
axes[0].legend(handles=legend_elements, loc='upper left')

plt.suptitle("Spatial Progression of Snow Squall - Monroe County", fontsize=16)
plt.tight_layout()
# plt.savefig("squall_progression_grid_cnn_only.png")

In [ ]:
# maybe logic ^ needs to be adjusted

# not severe snow (bc that could be the ground) maybe needs to be severe snow and obs

In [ ]:
# FINAL THOUGHTS:  2. The "No Spatial Pattern" Mystery (The Atmospheric Science Angle) --  The Lake Effect "Firehose"
# Monroe County (Rochester) sits right on Lake Ontario. A massive percentage of Snow Squall Warnings issued by WFO Buffalo for Monroe County are not for moving synoptic fronts; they are for intense, stationary lake-effect snow bands.
# If a 15-mile-wide lake-effect band sets up directly over downtown Rochester and just dumps snow for three hours, you will never see a sweeping spatial progression. You will just see the entire downtown camera cluster light up simultaneously and stay lit, pulsing in intensity. That is exactly what your maps look like.

# Could add weather data to confirm this if i want (but will shift to tryung to find a better more standard events rather than co-existant w snow squall) can chekc this w mrms data plot and/or ncei events saying these times are L-E or blizzards

In [ ]:
# Back to the events to find a moving synoptic front not instense stationary snow from LE....

super event Albany county

In [ ]:
display(top_coverage_events[40:60])

# albany 2024-01-14  eventid 5,6,8, MAYBE 15? or too much later?



In [ ]:
import pandas as pd

# 1. Define the parameters for your "Super Event"
target_ugc = "NYC001"

# We know the first event issued at 06:25 and the last at 08:17. 
# Let's grab everything from 05:00 to 10:00 to capture all buffers and the full decay.
start_time = "2024-01-14 15:00:00"
end_time = "2024-01-14 21:00:00"

# 2. Convert df_final timestamps to actual datetime objects if they aren't already
if not pd.api.types.is_datetime64_any_dtype(df_final['timestamp']):
    df_final['datetime'] = pd.to_datetime(df_final['timestamp'], format='%Y%m%d_%H%M')

# 3. Slice the dataframe using time and location, ignoring eventid
super_event_df = df_final[
    (df_final['ugc_gis'] == target_ugc) & 
    (df_final['datetime'] >= pd.to_datetime(start_time)) &
    (df_final['datetime'] <= pd.to_datetime(end_time))
].copy()

# Sort it chronologically just to be safe
super_event_df = super_event_df.sort_values('datetime')

# 4. Rebuild the Squall Hit logic for this new isolated dataframe
obs_class = 'obs'
snow_class = 'snow_severe'
poor_viz_class = 'poor_viz' 

super_event_df['hit_severe_snow'] = (super_event_df['select'] == snow_class).astype(int)
super_event_df['hit_poor_viz'] = (super_event_df['select'] == poor_viz_class).astype(int)
super_event_df['hit_odm_obs'] = (super_event_df['odm_select'] == obs_class).astype(int)

# Master squall hit flag
super_event_df['squall_hit'] = (super_event_df['hit_severe_snow'] | super_event_df['hit_odm_obs'])

print(f"Super Event extracted! Total 5-minute camera snapshots: {len(super_event_df):,}")

super_event_df.head(5)

In [ ]:
super_event_df[((super_event_df["ID"]=="Skyline_6053")&(super_event_df["timestamp"]=="20240114_1950"))]

In [ ]:
# 1. Create specific binary flags BEFORE aggregating
# (Adjust the string names to exactly match your model's output classes)
snow_class = 'snow_severe'
poor_viz_class = 'poor_viz' 
obs_class = 'obs'

super_event_df['hit_severe_snow'] = (super_event_df['select'] == snow_class).astype(int)
super_event_df['hit_poor_viz'] = (super_event_df['select'] == poor_viz_class).astype(int)
super_event_df['hit_odm_obs'] = (super_event_df['odm_select'] == obs_class).astype(int)

# 2. Aggregate data by time, now including our new columns
temporal_stats = super_event_df.groupby(['datetime', 'timetype']).agg(
    total_active_cams=('ID', 'count'),
    total_hits=('squall_hit', 'sum'),
    total_severe_snow=('hit_severe_snow', 'sum'),
    total_poor_viz=('hit_poor_viz', 'sum'),
    total_odm_obs=('hit_odm_obs', 'sum')
).reset_index()

# 3. Calculate the percentages for all categories
temporal_stats['percent_impacted'] = (temporal_stats['total_hits'] / temporal_stats['total_active_cams']) * 100
temporal_stats['percent_severe_snow'] = (temporal_stats['total_severe_snow'] / temporal_stats['total_active_cams']) * 100
temporal_stats['percent_poor_viz'] = (temporal_stats['total_poor_viz'] / temporal_stats['total_active_cams']) * 100
temporal_stats['percent_odm_obs'] = (temporal_stats['total_odm_obs'] / temporal_stats['total_active_cams']) * 100

# 4. Plotting
plt.figure(figsize=(14, 6))

# Plot the original combined "Master" line (made thicker/dashed to stand out)
# sns.lineplot(data=temporal_stats, x='datetime', y='percent_impacted', 
#              linewidth=3, color='black', linestyle='--', label='Combined Squall Hit (Any)')

# Plot the 3 specific class lines
sns.lineplot(data=temporal_stats, x='datetime', y='percent_severe_snow', 
             linewidth=2, color='red', marker = 'o', label=f'RSC: {snow_class}')
sns.lineplot(data=temporal_stats, x='datetime', y='percent_poor_viz', 
             linewidth=2, color='blue', marker = 'o', label=f'RSC: {poor_viz_class}')
sns.lineplot(data=temporal_stats, x='datetime', y='percent_odm_obs', 
             linewidth=1, color='gray', marker = 'o',  linestyle='--', label=f'ODM: {obs_class}')

# Visually separate the "Buffer" times from the "Event" times
event_start = temporal_stats[temporal_stats['timetype'] == 'event']['datetime'].min()
event_end = temporal_stats[temporal_stats['timetype'] == 'event']['datetime'].max()
# plt.axvspan(event_start, event_end, color='red', alpha=0.1, label='NWS Warning Active')

# # Draw a red box for each of the warning issuances in this time window
# unique_events = super_event_df[super_event_df['timetype'] == 'event']['eventid'].unique()

# # Uncomment this to show the Squall warning shading
# for eid in unique_events:
#     # Find the start and end of this specific eventid
#     evt_data = super_event_df[(super_event_df['eventid'] == eid) & (super_event_df['timetype'] == 'event')]
#     if not evt_data.empty:
#         e_start = evt_data['datetime'].min()
#         e_end = evt_data['datetime'].max()
#         plt.axvspan(e_start, e_end, color='red', alpha=0.15, label=f'NWS Squall Warning' if eid == unique_events[0] else "")
        
# plt.title(f"2024 Jan 14 Snow Squall Warning in Albany County - Model Predictions", fontsize=20)
# plt.xlabel("Time (UTC)", fontsize=16)
# plt.ylabel("Percentage of Cameras (%)", fontsize=16)
# plt.legend(fontsize=12)
# plt.grid(True, linestyle='--', alpha=0.6)
# plt.tight_layout()
# # plt.savefig("monroe_hazard_footprint_detailed.png")



# 1. Extract the absolute NWS warning times directly from the dataframe
true_warnings = super_event_df[['iso_issued', 'iso_expired']].drop_duplicates().dropna()

# 2. Loop through them and draw the background boxes
for _, row in true_warnings.iterrows():
    # Convert the ISO strings to pandas datetimes so they map to the X-axis properly
    issue_time = pd.to_datetime(row['iso_issued'])
    expire_time = pd.to_datetime(row['iso_expired'])
    
    # Plot the red warning block
    plt.axvspan(issue_time, expire_time, color='red', alpha=0.1, label='NWS Squall Warnings')

plt.title(f"2024 Jan 14 Snow Squall Warning in Albany County - Model Predictions", fontsize=20)
plt.xlabel("Time (UTC)", fontsize=16)
plt.ylabel("Percentage of Cameras (%)", fontsize=16)

# ==========================================
# CUSTOM LEGEND DEDUPLICATION
# ==========================================
# Grab all the labels, zip them into a dictionary (which automatically overwrites duplicates),
# and build the final legend with the fontsize included here!
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys(), fontsize=12)

plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
# plt.savefig("monroe_hazard_footprint_detailed.png")




In [ ]:
# # to save out, don't need to run again
super_event_df.to_csv("/home/csutter/DRIVE-clean/weather_events/data/squall_casestudies/albany/squall20240114_preds.csv")


In [ ]:
print(np.unique(super_event_df["iso_issued"]))
print(np.unique(super_event_df["iso_expired"]))

In [ ]:
# 1. Rename the column (Just once!)
# We use errors='ignore' so it doesn't crash if you run the cell twice
super_event_df = super_event_df.rename(columns={"geometry": "geometry_campoint"}, errors="ignore")

# 2. Convert text to spatial objects
# We add a quick check to make sure we only apply wkt.loads if it's still text
if isinstance(super_event_df['geometry_campoint'].iloc[0], str):
    super_event_df['geometry_campoint'] = super_event_df['geometry_campoint'].apply(wkt.loads)

# 3. THE MISSING LINK: Promote it from a regular Pandas DataFrame to a GeoDataFrame!
super_event_df = gpd.GeoDataFrame(super_event_df, geometry="geometry_campoint")

# 4. Load the relevant County shapefile safely
county_shape = devents[devents['ugc_gis'] == target_ugc].iloc[0:1]

# 5. Test the plot!

# 1. First, explicitly tell GeoPandas that our current numbers are standard GPS Lat/Lon (EPSG:4326)
if super_event_df.crs is None:
    super_event_df = super_event_df.set_crs("EPSG:4326")
if county_shape.crs is None:
    county_shape = county_shape.set_crs("EPSG:4326")

# 2. Re-project both dataframes to Web Mercator (EPSG:3857) to fix the stretching
super_event_df = super_event_df.to_crs("EPSG:3857")
county_shape = county_shape.to_crs("EPSG:3857")

ax = county_shape.plot(color='white', edgecolor='black', figsize=(8, 8))
super_event_df.plot(ax=ax, color='lightgrey', markersize=10)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from shapely import wkt

# ==========================================
# 1. DEFINE REGION & EXTRACT DATA
# ==========================================
countiesuse = [
    "NYC001", "NYC093", "NYC083", "NYC091", "NYC095", "NYC039", "NYC021", 
    "NYC057", "NYC035", "NYC043", "NYC077", "NYC115", "NYC113", "NYC025", 
    "NYC111", "NYC027"
]

start_time_zoom = "2024-01-14 16:25:00"
end_time_zoom = "2024-01-14 20:00:00"

# Pull the regional data from your MASTER dataframe
regional_df = df_final[
    (df_final['ugc_gis'].isin(countiesuse)) & 
    (df_final['datetime'] >= start_time_zoom) & 
    (df_final['datetime'] <= end_time_zoom)
].copy()



# Apply the strict CNN hit flags using the correct class names
regional_df['hit_severe_snow'] = (regional_df['select'] == 'snow_severe').astype(int)
regional_df['hit_poor_viz'] = (regional_df['select'] == 'poor_viz').astype(int)

# Set the active geometry so GeoPandas knows how to map the cameras
# regional_df = regional_df.rename(columns = {"geometry":"geometry_campoint"})
# regional_df = regional_df.set_geometry("geometry_campoint")

# Set the active geometry so GeoPandas knows how to map the cameras
regional_df = regional_df.rename(columns = {"geometry":"geometry_campoint"})

# ---> THE FIX: Convert the text strings back into true spatial objects <---
regional_df['geometry_campoint'] = regional_df['geometry_campoint'].apply(wkt.loads)

# Now GeoPandas will happily accept it!
regional_df = regional_df.set_geometry("geometry_campoint")


# Create the background map using those 16 counties
regional_shape = devents[devents['ugc_gis'].isin(countiesuse)].drop_duplicates(subset=['ugc_gis'])

# ==========================================
# 2. TIME SLICING (16 EVEN INTERVALS)
# ==========================================
# 1. Get the sorted list of all available times in this new window
unique_times = sorted(regional_df['datetime'].unique())

# 2. Mathematically calculate 16 evenly spaced indices to fit a 4x4 grid
indices = np.linspace(0, len(unique_times) - 1, 16).astype(int)

# 3. Pull those specific times
times_to_plot = [unique_times[i] for i in indices]

# ==========================================
# 3. PLOTTING THE 4x4 GRID
# ==========================================
fig, axes = plt.subplots(4, 4, figsize=(24, 18)) # Slightly taller figsize to handle the massive NY state region
axes = axes.flatten()

for i, ts in enumerate(times_to_plot):
    ax = axes[i]
    
    # 1. Plot the multi-county outline
    regional_shape.plot(ax=ax, color='white', edgecolor='black', linewidth=1)
    
    # Extract data for this exact timestamp
    time_data = regional_df[regional_df['datetime'] == ts]
    
    # 2. Plot BASE cameras (No Severe Snow and No Poor Viz)
    base_cams = time_data[(time_data['hit_severe_snow'] == 0) & (time_data['hit_poor_viz'] == 0)]
    if not base_cams.empty:
        # Changed to whitesmoke fill with a black outline
        base_cams.plot(ax=ax, color='whitesmoke', edgecolor='black', markersize=20)
    
    # 3. Plot POOR VIZ hits (Excluding Severe Snow so they don't plot twice)
    pv_cams = time_data[(time_data['hit_poor_viz'] == 1) & (time_data['hit_severe_snow'] == 0)]
    if not pv_cams.empty:
        pv_cams.plot(ax=ax, color='blue', edgecolor='black', markersize=60)
        
    # 4. Plot SEVERE SNOW hits (Takes top priority)
    ss_cams = time_data[time_data['hit_severe_snow'] == 1]
    if not ss_cams.empty:
        ss_cams.plot(ax=ax, color='red', edgecolor='black', markersize=60)
    
    # Format time
    formatted_time = pd.to_datetime(ts).strftime('%H:%M')
    ax.set_title(f"Time: {formatted_time} UTC", fontsize=14)
    ax.axis('off')

# ==========================================
# 4. CUSTOM LEGEND & CLEANUP
# ==========================================
legend_elements = [
    mlines.Line2D([0], [0], marker='o', color='w', markerfacecolor='whitesmoke', markeredgecolor='black', markersize=10, label='No hazard'),
    mlines.Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', markeredgecolor='black', markersize=12, label='CNN: poor_viz'),
    mlines.Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markeredgecolor='black', markersize=12, label='CNN: snow_severe')
]

axes[0].legend(handles=legend_elements, loc='upper left', fontsize=12)

# Updated the title to reflect the new geography!
plt.suptitle("Spatial Progression of Winter Hazard - Eastern NY", fontsize=22, y=1.02)
plt.tight_layout()

# plt.savefig("eastern_ny_hazard_progression.png", bbox_inches='tight')

In [ ]:
# to save out, don't need to run again
regional_df.to_csv("/home/csutter/DRIVE-clean/weather_events/data/squall_casestudies/albany/regional_squall20240114_preds.csv")


In [ ]:
regional_df[regional_df[]]

In [ ]:
# check a few of those cams. were they actually obs??

regional_df[50:55]


# Cams randomly pulling that weren't "no live feed" but then switch to it during the event???
# /home/csutter/cron/data/Skyline_6527/20240114/I_90_at_Interchange_29A_(Little_Falls)__Westbound__Skyline_6527_2024-01-14-17:03:30.jpg

# /home/csutter/cron/data/Skyline_6528/20240114/I_90_between_Interchange_29A_(Little_Falls)_and_Interchange_30_(Herkimer)__Westbound__Skyline_6528_2024-01-14-16:14:32.jpg

# /home/csutter/cron/data/Skyline_5992/20240114/Route_30_High_Mast_#1_(Amsterdam)__Northbound__Skyline_5992_2024-01-14-15:00:18.jpg

# /home/csutter/cron/data/Skyline_6820/20240114/Rte_30_at_School_St_Mayfield__Southbound__Skyline_6820_2024-01-14-15:53:11.jpg

# /home/csutter/cron/data/Skyline_2106/20240114/Route_30A_at_Route_67___Johnstown__Northbound__Skyline_2106_2024-01-14-16:20:18.jpg

# /home/csutter/cron/data/Skyline_6421/20240114/I_87_Just_North_of_Interchange_19_(Kingston)__Northbound__Skyline_6421_2024-01-14-16:21:45.jpg <--- but this site had good one giving PV as the RSC when it was PV!

# Can't really DO anything with the obs class.... It's not reliable without having labeled them. Thus cant really do anything with the squalls , unless we rely on PV alone. 


In [ ]:
nonobs = regional_df[regional_df["odm_select"]=="nonobs"]
display(nonobs[30:50])

In [ ]:
# See which ones are poor vis

regional_df[regional_df["squall_hit"]==1].head(3)

In [ ]:
regional_df[regional_df["ID"]=='Skyline_2085']

In [ ]:
# Next idea: let's "zoom out" of just plotting albany, and plot the entire region so that we can see all camera preds changing over a broader region -- this may give better squall results! looking at just within county too small to capture deltas, since the squall may have impacted everywhere in that county


# if snow aqualls are hard to capture w met data (which is our argument of using cams/our model) then how can i layer on weather data to "see" the squall?

# need to label some images from these events?

In [ ]:
# the baseline of 25% being obstructed ---- cameras just not active!! need to look at squall RELATIVE to baseline obs amount. but why the dips down during the event? Weird, would imply to me those arent inactive cams. 

# For the other analyses (since we didnt actually remove obs examples, wouldnt want to) those inactive cams aRE in our stats analyses. They are likely leaning on the HRRR data.... check that out, whether obs cases are low prob and are heavily influenced by weather data addition

### Img analysis - cases from the above squalls

Read in dfs already saved from above

In [ ]:
monroe = pd.read_csv("/home/csutter/DRIVE-clean/weather_events/data/squall_casestudies/monroe/squall20250207_preds.csv")

albany = pd.read_csv("/home/csutter/DRIVE-clean/weather_events/data/squall_casestudies/albany/squall20240114_preds.csv")

albregion = pd.read_csv("/home/csutter/DRIVE-clean/weather_events/data/squall_casestudies/albany/regional_squall20240114_preds.csv")

In [ ]:
# Monroe
# Skyline_5042 and Skyline_1811

Skyline_5042 = monroe[monroe["ID"]=="Skyline_5042"]
# display(Skyline_5042.head(3))

Skyline_1811 = monroe[monroe["ID"]=="Skyline_1811"]

Skyline_5046 = monroe[monroe["ID"]=="Skyline_5046"]


In [ ]:
monroe.head(4)

exch = monroe[monroe['Name'].str.contains('Exchange', na=False)]
exch.head(4)

In [ ]:
### Albany 
# Skyline_6425 and Skyline_6053
Skyline_6425 = albany[albany["ID"]=="Skyline_6425"]
Skyline_6053 = albany[albany["ID"]=="Skyline_6053"]


### Albany REGION
# Skyline_6421 
Skyline_6421 = albregion[albregion["ID"]=="Skyline_6421"]


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Custom color palette based on your image
custom_colors = {
    'dry': '#71c249',           # Green from the image
    'wet': '#70a8db',           # Blue from the image
    'snow': 'pink',          # Pink from the image
    'snow_severe': '#eb3324',   # Red from the image
    'poor_viz': '#4a148c'       # A distinct purple
}

def plot_predictions_and_print_table(df, maintitle, subtitle, color_map = custom_colors):
    """
    Plots the prediction probability over time with custom color-coded 
    classification dots, adds a dotted line for poor visibility probability, 
    and prints a table of timestamps and image paths.
    """
    # Create the figure
    fig, ax = plt.subplots(figsize=(12, 6))

    # 1. Plot the connected line for select_prob (all one color)
    ax.plot(df['timestamp'], df['select_prob'], color='gray', linestyle='-', 
            zorder=1, label='Predicted Class Prob')

    # 2. Plot the dotted line for poor visibility probability
    # We pull the specific purple color directly from your color map
    ax.plot(df['timestamp'], df['ensembleAvg_poor_viz'], color=color_map['poor_viz'], 
            linestyle=':', linewidth=2, zorder=1, label='Poor Visibility Prob')

    # 3. Plot the large, custom color-coded dots for the predicted class
    sns.scatterplot(
        data=df, 
        x='timestamp', 
        y='select_prob', 
        hue='select',   
        s=200,          
        palette=color_map, # <-- This forces Seaborn to use your exact colors
        edgecolor='black',
        ax=ax,
        zorder=2
    )

    # Formatting the plot
    plt.suptitle(maintitle, fontsize=16)
    plt.title(subtitle, fontsize=14)
    plt.xlabel('Timestamp', fontsize=12)
    plt.ylabel('Probability', fontsize=12)
    
    plt.xticks(rotation=45, ha='right')
    # plt.legend(bbox_to_anchor=(1, 1), loc='upper left')
    plt.legend(bbox_to_anchor=(0, .15), loc='lower left')
    
    plt.tight_layout()
    plt.show()

    # Print the table of timestamps and image paths
    print("\n" + "="*50)
    print("TIMESTAMPS AND CORRESPONDING IMAGE PATHS")
    print("="*50)
    
    table_df = df[['timestamp', 'img_orig']]
    print(table_df.to_string(index=False))

# Run the function, passing in your cleaned dataframe and your new color map
# plot_predictions_and_print_table(df_clean, custom_colors)

Monroe

In [ ]:
plot_predictions_and_print_table(Skyline_5042, maintitle = 'Snow Squall Event in Rochester, NY', subtitle = 'Model Pediction and Probabilities for Camera: Skyline_5042') # this one prediction PV, so it's a little more cohesive, but the broader PV argument still works in other examples

In [ ]:
Skyline_5042.head(3)

In [ ]:
# Open the image
img = Image.open('/home/csutter/cron/data/Skyline_5042/20250207/West_Main_St_at_South_Plymouth_Ave__Unknown__Skyline_5042_2025-02-07-07:01:24.jpg')
img.show() 

img = Image.open('/home/csutter/cron/data/Skyline_5042/20250207/West_Main_St_at_South_Plymouth_Ave__Unknown__Skyline_5042_2025-02-07-07:26:17.jpg')
img.show() 



img = Image.open('/home/csutter/cron/data/Skyline_5042/20250207/West_Main_St_at_South_Plymouth_Ave__Unknown__Skyline_5042_2025-02-07-07:40:59.jpg')
img.show() 



img = Image.open('/home/csutter/cron/data/Skyline_5042/20250207/West_Main_St_at_South_Plymouth_Ave__Unknown__Skyline_5042_2025-02-07-08:03:25.jpg')
img.show() 


img = Image.open('/home/csutter/cron/data/Skyline_5042/20250207/West_Main_St_at_South_Plymouth_Ave__Unknown__Skyline_5042_2025-02-07-08:56:37.jpg')
img.show() 


In [ ]:
plot_predictions_and_print_table(Skyline_1811, maintitle = 'Snow Squall Event in Rochester, NY', subtitle = 'Model Pediction and Probabilities for Camera: Skyline_1811') # here, the model predicted Dry, but look at the dip in confidence (~30%) and also the bump in poor visibility prob! Not enough to make it predict PV, but the signature is still there. 

In [ ]:
plot_predictions_and_print_table(Skyline_5046, maintitle = 'Snow Squall Event in Rochester, NY', subtitle = 'Model Pediction and Probabilities for Camera: Skyline_5046') # This the one that wrongly predicted wet

In [ ]:
img = Image.open('/home/csutter/cron/data/Skyline_5046/20250207/Exchange_Blvd_at_Court_St__Unknown__Skyline_5046_2025-02-07-07:13:41.jpg')
img.show() 

Albany

In [ ]:
plot_predictions_and_print_table(Skyline_6053, maintitle = 'Snow Squall Event in Albany, NY', subtitle = 'Model Pediction and Probabilities for Camera: Skyline_6053') # state office campus

In [ ]:
img = Image.open('/home/csutter/cron/data/Skyline_6053/20240114/NY_85_at_the_State_Office_Campus__Eastbound__Skyline_6053_2024-01-14-16:01:08.jpg')
img.show() 

img = Image.open('/home/csutter/cron/data/Skyline_6053/20240114/NY_85_at_the_State_Office_Campus__Eastbound__Skyline_6053_2024-01-14-17:31:44.jpg')
img.show() 

img = Image.open('/home/csutter/cron/data/Skyline_6053/20240114/NY_85_at_the_State_Office_Campus__Eastbound__Skyline_6053_2024-01-14-19:30:41.jpg')
img.show() 

img = Image.open('/home/csutter/cron/data/Skyline_6053/20240114/NY_85_at_the_State_Office_Campus__Eastbound__Skyline_6053_2024-01-14-19:58:26.jpg')
img.show() 

img = Image.open('/home/csutter/cron/data/Skyline_6053/20240114/NY_85_at_the_State_Office_Campus__Eastbound__Skyline_6053_2024-01-14-20:08:21.jpg')
img.show() 

img = Image.open('/home/csutter/cron/data/Skyline_6053/20240114/NY_85_at_the_State_Office_Campus__Eastbound__Skyline_6053_2024-01-14-20:26:31.jpg')
img.show() 

In [ ]:
plot_predictions_and_print_table(Skyline_6425, maintitle = 'Snow Squall Event in Albany, NY', subtitle = 'Model Pediction and Probabilities for Camera: Skyline_6425') # highway Albany county

In [ ]:
# img = Image.open('/home/csutter/cron/data/Skyline_6425/20240114/I_87_at_Interchange_21A_(Berkshire_Connector)__Northbound__Skyline_6425_2024-01-14-17:24:34.jpg')
# img.show() 

img = Image.open('/home/csutter/cron/data/Skyline_6425/20240114/I_87_at_Interchange_21A_(Berkshire_Connector)__Northbound__Skyline_6425_2024-01-14-18:05:13.jpg')
img.show() 

img = Image.open('/home/csutter/cron/data/Skyline_6425/20240114/I_87_at_Interchange_21A_(Berkshire_Connector)__Northbound__Skyline_6425_2024-01-14-17:24:34.jpg')
img.show() 

img = Image.open('/home/csutter/cron/data/Skyline_6425/20240114/I_87_at_Interchange_21A_(Berkshire_Connector)__Northbound__Skyline_6425_2024-01-14-19:16:06.jpg')
img.show() 

img = Image.open('/home/csutter/cron/data/Skyline_6425/20240114/I_87_at_Interchange_21A_(Berkshire_Connector)__Northbound__Skyline_6425_2024-01-14-19:22:37.jpg')
img.show() 

img = Image.open('/home/csutter/cron/data/Skyline_6425/20240114/I_87_at_Interchange_21A_(Berkshire_Connector)__Northbound__Skyline_6425_2024-01-14-19:37:52.jpg')
img.show() 

img = Image.open('/home/csutter/cron/data/Skyline_6425/20240114/I_87_at_Interchange_21A_(Berkshire_Connector)__Northbound__Skyline_6425_2024-01-14-19:52:35.jpg')
img.show() 

In [ ]:
Skyline_6425.head(3)

In [ ]:
Skyline_6421.head(4)

In [ ]:
Skyline_6421_b = Skyline_6421[Skyline_6421.index > 2896]

plot_predictions_and_print_table(Skyline_6421_b, maintitle = 'Snow Squall Event in Ulster, NY', subtitle = 'Model Pediction and Probabilities for Camera: Skyline_6421') # Regional (Ulster county, but they use ALY WFO)

display(Skyline_6421.head(3))

In [ ]:


img = Image.open('/home/csutter/cron/data/Skyline_6421/20240114/I_87_Just_North_of_Interchange_19_(Kingston)__Northbound__Skyline_6421_2024-01-14-18:40:39.jpg')
img.show() 

In [ ]:



img = Image.open('/home/csutter/cron/data/Skyline_6421/20240114/I_87_Just_North_of_Interchange_19_(Kingston)__Northbound__Skyline_6421_2024-01-14-18:45:50.jpg')
img.show() 

end

In [ ]:
print(np.unique(top_coverage_events["ugc_gis"]))

In [ ]:
event_stats.head(100)